# 02z — Build Master Table (cierre Fase 1, pipeline v2)

Consolida los 8 parquets de features (02a/02c/02e/02f/02g/02i/02l/02m) en una
única master table.

**Pipeline v2**: 02b (currency), 02d (arena) y 02j (fights) están EXCLUIDOS por
CSVs truncados a 30d (ver `informes/fase1_cleaning/notebooks_excluded_v2.md`).
Descartes definitivos (02h/02k/02n) en `informes/decision_descarte.md`.

**Responsabilidad acotada**: este notebook **construye y diagnostica**.
La eliminación de features redundantes/cuasi-constantes va en `02zz_master_cleanup`.

## Adaptaciones vs prompt original (documentadas tras sondeo de parquets)

| Aspecto | Realidad detectada | Adaptación aplicada |
|---|---|---|
| Targets | `churn_14d`, `churn_30d` (int64) en `sample_user_ids.parquet` Y `users_clean.parquet` | Usar nombres reales; mantener los del sample, dropear duplicados de users_clean |
| Prefijo 02b | mix: `tx_*`, `gold_*`, `gems_*`, `dark_steel_*`, `concept_*`, `other_*` | NO renombrar — cada feature ya lleva contexto. Bloque "currency" agrupa todas |
| Prefijo 02e | `reward_*` (no `daily_*`) | usar real |
| Prefijo 02g | `device_*` (no `dev_*`) | usar real |
| Prefijo 02i | `feedback_*` (no `supp_*`) | usar real |
| Prefijo 02j | `fights_*` (con s, no `fight_*`) | usar real |
| `fights_first_dt`, `fights_last_dt` | Int64 nullable unix seconds (no datetime) | convertir vía `pd.to_datetime(unit='s')` por nombre |
| `features_fights.BROKEN.parquet` | Existe en data_qc, NO usar | lista explícita de inputs |

## Inputs (9 parquets verificados, pipeline v2)

- `sample_user_ids_cutoff.parquet` (base; ya contiene `churn_14d`, `churn_30d`)
- `users_clean_cutoff.parquet` (02a; metadata de usuario)
- ~~`features_currency_cutoff.parquet`~~ (02b — EXCLUIDO v2)
- `features_characters_cutoff.parquet` (02c; `char_*`)
- ~~`features_arena_cutoff.parquet`~~ (02d — EXCLUIDO v2)
- `features_daily_rewards_cutoff.parquet` (02e; `reward_*`)
- `features_iaps_cutoff.parquet` (02f; `iap_*`)
- `features_devices_cutoff.parquet` (02g; `device_*`)
- `features_feedback_cutoff.parquet` (02i; `feedback_*`)
- ~~`features_fights_cutoff.parquet`~~ (02j — EXCLUIDO v2)
- `features_user_items_collection_cutoff.parquet` (02l; `coll_*`)
- `features_user_items_cutoff.parquet` (02m; `items_*`)

## Outputs
- `data/data_qc/master_table.parquet`
- `data/data_qc/master_table_schema.json`
- `informes/fase1_cleaning/master_table/` (ver criterios de aceptación)


In [1]:
# [SETUP] Imports y dependencias
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc
import time
import json
from pathlib import Path
from datetime import datetime

plt.style.use('default')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 240)


In [2]:
# [SETUP] Paths, constantes y helpers
PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase1_cleaning' else Path.cwd()
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
REPORT_DIR = PROJECT_ROOT / 'informes' / 'fase1_cleaning' / 'master_table'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_DATE = pd.Timestamp('2026-04-04', tz='UTC')
CUTOFF_DATE = REFERENCE_DATE - pd.Timedelta(days=120)
SAMPLE_PATH = DATA_QC / 'sample_user_ids_cutoff.parquet'
# N_SAMPLE leído dinámicamente del sample (con cutoff cambia)
N_SAMPLE = len(pd.read_parquet(SAMPLE_PATH, columns=['user_id']))
MASTER_PARQUET = DATA_QC / 'master_table_cutoff.parquet'
MASTER_SCHEMA = DATA_QC / 'master_table_cutoff_schema.json'

# Lista de parquets a integrar, en orden de JOIN.
# Cada entrada: (nombre lógico del bloque, path, prefijo principal, fuente notebook)
# Pipeline v2: 02b/02d/02j EXCLUIDOS por CSVs truncados a 30d.
# Ver informes/fase1_cleaning/notebooks_excluded_v2.md
PARQUET_INPUTS = [
    ('users',     DATA_QC / 'users_clean_cutoff.parquet',                       'user',     '02a'),
    ('char',      DATA_QC / 'features_characters_cutoff.parquet',               'char',     '02c'),
    ('reward',    DATA_QC / 'features_daily_rewards_cutoff.parquet',            'reward',   '02e'),
    ('iap',       DATA_QC / 'features_iaps_cutoff.parquet',                     'iap',      '02f'),
    ('device',    DATA_QC / 'features_devices_cutoff.parquet',                  'device',   '02g'),
    ('feedback',  DATA_QC / 'features_feedback_cutoff.parquet',                 'feedback', '02i'),
    ('coll',      DATA_QC / 'features_user_items_collection_cutoff.parquet',    'coll',     '02l'),
    ('items',     DATA_QC / 'features_user_items_cutoff.parquet',               'items',    '02m'),
]

steps_log = []
NOTEBOOK_START = time.time()

def log_step(tag, message):
    ts = datetime.now().strftime('%H:%M:%S')
    entry = f"[{tag}] {ts} — {message}"
    steps_log.append(entry)
    print(entry)

# Mapeo de columna → bloque (se rellena durante los JOINs)
column_to_block = {}

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"REFERENCE_DATE: {REFERENCE_DATE}")
print(f"N_SAMPLE: {N_SAMPLE:,}")
print(f"\nParquets a integrar: {len(PARQUET_INPUTS)} bloques + sample base")


PROJECT_ROOT: /Users/jezquerro/Documents/tfg
REFERENCE_DATE: 2026-04-04 00:00:00+00:00
N_SAMPLE: 25,200

Parquets a integrar: 8 bloques + sample base


## BLOQUE 2 — Verificación de inputs

In [3]:
# [EXEC] 2.1 Existencia + 2.2 schema + 2.4 dtype de user_id
input_summary = []

for block, path, prefix, source in PARQUET_INPUTS:
    if not path.exists():
        raise FileNotFoundError(f'STOP: parquet faltante para bloque {block}: {path}')
    df = pd.read_parquet(path)
    has_user_id = 'user_id' in df.columns
    user_id_unique = bool(df['user_id'].is_unique) if has_user_id else False
    user_id_dtype = str(df['user_id'].dtype) if has_user_id else 'MISSING'
    input_summary.append({
        'block': block,
        'source': source,
        'path': path.name,
        'rows': len(df),
        'cols': df.shape[1],
        'has_user_id': has_user_id,
        'user_id_unique': user_id_unique,
        'user_id_dtype': user_id_dtype,
    })
    del df
    gc.collect()

# Sample (base)
df_sample = pd.read_parquet(SAMPLE_PATH)
input_summary.insert(0, {
    'block': 'sample_base',
    'source': '02a',
    'path': SAMPLE_PATH.name,
    'rows': len(df_sample),
    'cols': df_sample.shape[1],
    'has_user_id': True,
    'user_id_unique': bool(df_sample['user_id'].is_unique),
    'user_id_dtype': str(df_sample['user_id'].dtype),
})

summary_df = pd.DataFrame(input_summary)
print(summary_df.to_string(index=False))

# Asserts duros: todos deben tener user_id y ser únicos
for row in input_summary:
    assert row['has_user_id'], f"STOP: {row['block']} sin user_id"
    assert row['user_id_unique'], f"STOP: {row['block']} con user_id duplicado"

log_step('EXEC', f'2.1 - 2.4: {len(input_summary)} parquets verificados, todos con user_id único')


      block source                                          path  rows  cols  has_user_id  user_id_unique user_id_dtype
sample_base    02a                sample_user_ids_cutoff.parquet 25200     3         True            True           str
      users    02a                    users_clean_cutoff.parquet 25200    30         True            True           str
       char    02c            features_characters_cutoff.parquet 25200    24         True            True           str
     reward    02e         features_daily_rewards_cutoff.parquet 25200    11         True            True           str
        iap    02f                  features_iaps_cutoff.parquet 25200    23         True            True           str
     device    02g               features_devices_cutoff.parquet 25200    11         True            True           str
   feedback    02i              features_feedback_cutoff.parquet 25200     7         True            True           str
       coll    02l features_user_items_c

In [4]:
# [ANALYSIS] 2.3 Auditoría de prefijos
prefix_audit = []
for block, path, prefix, source in PARQUET_INPUTS:
    df = pd.read_parquet(path)
    cols = [c for c in df.columns if c != 'user_id']
    if prefix is None:
        # currency: mix de prefijos, no hay un único esperado
        prefix_audit.append({
            'block': block,
            'expected_prefix': '(mix)',
            'total_cols': len(cols),
            'with_prefix': 'N/A',
            'without_prefix': 'N/A',
            'note': 'Cada col lleva su propio contexto (tx_, gold_, gems_, ...)',
        })
    else:
        with_p = sum(1 for c in cols if c.startswith(f'{prefix}_'))
        without_p = len(cols) - with_p
        prefix_audit.append({
            'block': block,
            'expected_prefix': f'{prefix}_*',
            'total_cols': len(cols),
            'with_prefix': with_p,
            'without_prefix': without_p,
            'note': 'OK' if without_p == 0 else f'{without_p} cols sin prefijo',
        })
    del df
    gc.collect()

prefix_df = pd.DataFrame(prefix_audit)
print(prefix_df.to_string(index=False))
log_step('ANALYSIS', '2.3 prefijos auditados — todos los notebooks con prefijo dedicado lo cumplen 100%')


   block expected_prefix  total_cols  with_prefix  without_prefix                note
   users          user_*          29            0              29 29 cols sin prefijo
    char          char_*          23           23               0                  OK
  reward        reward_*          10           10               0                  OK
     iap           iap_*          22           22               0                  OK
  device        device_*          10           10               0                  OK
feedback      feedback_*           6            6               0                  OK
    coll          coll_*           8            8               0                  OK
   items         items_*          13           13               0                  OK
[ANALYSIS] 13:16:59 — 2.3 prefijos auditados — todos los notebooks con prefijo dedicado lo cumplen 100%


## BLOQUE 3 — Construcción de la master (JOINs encadenados)

Estrategia: LEFT JOIN siempre desde `sample_user_ids.parquet`. El sample manda.
Si tras un JOIN el número de filas crece, hay duplicados → STOP.


In [5]:
# [EXEC] 3.1 Punto de partida: sample_user_ids
master = pd.read_parquet(SAMPLE_PATH)
N_SAMPLE = len(master)  # leído dinámicamente del sample (con cutoff cambia)
assert master['user_id'].is_unique, 'sample tiene user_id duplicados'
assert N_SAMPLE > 20_000, f'Sample sospechosamente pequeño: {N_SAMPLE}'
print(f'N_SAMPLE = {N_SAMPLE:,}')

# Trackear bloque por columna
for c in master.columns:
    if c == 'user_id':
        column_to_block[c] = 'key'
    elif c.startswith('churn_'):
        column_to_block[c] = 'target'
    else:
        column_to_block[c] = 'sample_base'

print(f'Master inicial: {master.shape}')
print(f'Columnas: {list(master.columns)}')
log_step('EXEC', f'3.1 master base: {master.shape}, cols={list(master.columns)}')


N_SAMPLE = 25,200
Master inicial: (25200, 3)
Columnas: ['user_id', 'churn_14d', 'churn_30d']
[EXEC] 13:16:59 — 3.1 master base: (25200, 3), cols=['user_id', 'churn_14d', 'churn_30d']


In [6]:
# [EXEC] 3.3 JOIN users_clean (caso especial)
# Política:
#   - Drop: _id, own_id (IDs internos redundantes con user_id)
#   - Drop: churn_14d, churn_30d (duplicados con sample, ya en master)
#   - NO renombrar (categóricas claras): country, language, is_google_play,
#     tutorial_done, has_user_rated_app, focus
#   - Renombrar a user_*: el resto

KEEP_AS_IS = {
    'country', 'language', 'is_google_play', 'tutorial_done',
    'has_user_rated_app', 'focus',
}
DROP_FROM_USERS = {'_id', 'own_id', 'churn_14d', 'churn_30d'}

df_users = pd.read_parquet(DATA_QC / 'users_clean_cutoff.parquet')
n_users_before_drop = df_users.shape[1]

# Drop columnas redundantes/internas
df_users = df_users.drop(columns=[c for c in DROP_FROM_USERS if c in df_users.columns])

# Renombrado: todo lo que no esté en KEEP_AS_IS y no sea user_id pasa a 'user_*'
rename_map = {}
for c in df_users.columns:
    if c == 'user_id':
        continue
    if c in KEEP_AS_IS:
        continue
    if c.startswith('user_'):
        continue  # ya renombrado
    rename_map[c] = f'user_{c}'

df_users = df_users.rename(columns=rename_map)
print(f'users_clean: shape original {n_users_before_drop} cols → {df_users.shape[1]} cols tras drop+rename')
print(f'  Dropped: {sorted(DROP_FROM_USERS & set(pd.read_parquet(DATA_QC / "users_clean_cutoff.parquet").columns))}')
print(f'  Renamed ({len(rename_map)}): {dict(list(rename_map.items())[:5])}{"..." if len(rename_map)>5 else ""}')
print(f'  Kept-as-is ({len(KEEP_AS_IS & set(df_users.columns))}): {sorted(KEEP_AS_IS & set(df_users.columns))}')

# JOIN
n_before = len(master)
master = master.merge(df_users, on='user_id', how='left')
assert len(master) == n_before, f'JOIN users_clean cambió shape: {n_before} → {len(master)}'
assert master['user_id'].is_unique, 'user_id duplicado tras JOIN users_clean'
assert master.columns.is_unique, 'colisión de nombres tras JOIN users_clean'

# Trackear bloque
for c in df_users.columns:
    if c == 'user_id':
        continue
    column_to_block[c] = 'users'

print(f'\nMaster tras JOIN users_clean: {master.shape}')
log_step('EXEC', f'JOIN users_clean: +{len(df_users.columns)-1} cols, shape={master.shape}')
del df_users
gc.collect()


users_clean: shape original 30 cols → 26 cols tras drop+rename
  Dropped: ['_id', 'churn_14d', 'churn_30d', 'own_id']
  Renamed (19): {'last_login_date': 'user_last_login_date', 'num_logins': 'user_num_logins', 'game_version': 'user_game_version', 'last_completed_tutorial_block': 'user_last_completed_tutorial_block', 'current_session': 'user_current_session'}...
  Kept-as-is (6): ['country', 'focus', 'has_user_rated_app', 'is_google_play', 'language', 'tutorial_done']

Master tras JOIN users_clean: (25200, 28)
[EXEC] 13:16:59 — JOIN users_clean: +25 cols, shape=(25200, 28)


0

In [7]:
# [EXEC] 3.2 LEFT JOIN secuencial del resto de bloques
# Saltamos 'users' (ya hecho con política especial)
for block, path, prefix, source in PARQUET_INPUTS:
    if block == 'users':
        continue

    df = pd.read_parquet(path)

    # Política de renombrado: si tiene prefijo dedicado y todas las cols lo respetan, no tocar.
    # Si NO tiene prefijo dedicado (currency), tampoco renombrar — cada col ya lleva contexto.
    # En otras palabras: NO renombramos en este loop, los notebooks de origen ya lo hicieron bien.

    n_before = len(master)
    n_cols_before = master.shape[1]
    master = master.merge(df, on='user_id', how='left')

    # Asserts duros
    assert len(master) == n_before, f'STOP: JOIN {block} cambió shape: {n_before} → {len(master)}'
    assert master['user_id'].is_unique, f'STOP: user_id duplicado tras JOIN {block}'
    assert master.columns.is_unique, f'STOP: colisión de nombres tras JOIN {block}'

    # Trackear bloque por columna
    new_cols = [c for c in df.columns if c != 'user_id']
    for c in new_cols:
        column_to_block[c] = block

    log_step('EXEC', f'JOIN {block} ({source}): +{len(new_cols)} cols, shape={master.shape}')
    del df
    gc.collect()

print(f'\n=== Master tras todos los JOINs ===')
print(f'Shape: {master.shape}')
print(f'Memoria: {master.memory_usage(deep=True).sum()/1e6:.1f} MB')


[EXEC] 13:16:59 — JOIN char (02c): +23 cols, shape=(25200, 51)
[EXEC] 13:16:59 — JOIN reward (02e): +10 cols, shape=(25200, 61)
[EXEC] 13:16:59 — JOIN iap (02f): +22 cols, shape=(25200, 83)


[EXEC] 13:16:59 — JOIN device (02g): +10 cols, shape=(25200, 93)


[EXEC] 13:16:59 — JOIN feedback (02i): +6 cols, shape=(25200, 99)
[EXEC] 13:16:59 — JOIN coll (02l): +8 cols, shape=(25200, 107)


[EXEC] 13:16:59 — JOIN items (02m): +13 cols, shape=(25200, 120)

=== Master tras todos los JOINs ===
Shape: (25200, 120)
Memoria: 25.1 MB


## BLOQUE 4 — Conversión de datetimes a numéricos

Cualquier columna datetime se convierte a `(REFERENCE_DATE - col).dt.days`.
También trato `fights_first_dt`/`fights_last_dt` (Int64 unix seconds) por nombre.
Detecto y dropeo redundancias contra columnas `*_days_since_*` ya derivadas.


In [8]:
# [EXEC] 4.1+4.2 Detectar e identificar columnas datetime / unix-int
datetime_cols = []
unix_int_cols = []

for col in master.columns:
    dtype = master[col].dtype
    # datetime con cualquier resolución/timezone
    if pd.api.types.is_datetime64_any_dtype(dtype):
        datetime_cols.append(col)
    # Int64/int64 con sufijo _dt, _at, _date → posible unix seconds
    elif col.endswith(('_dt', '_at', '_date')) and (
        pd.api.types.is_integer_dtype(dtype) or str(dtype) == 'Int64'
    ):
        unix_int_cols.append(col)

print(f'Columnas datetime detectadas: {len(datetime_cols)}')
for c in datetime_cols:
    print(f'  {c} (dtype={master[c].dtype}, block={column_to_block.get(c)})')
print(f'\nColumnas unix-int detectadas (sufijo *_dt/_at/_date + dtype int): {len(unix_int_cols)}')
for c in unix_int_cols:
    print(f'  {c} (dtype={master[c].dtype}, block={column_to_block.get(c)})')


def derive_days_ago_name(col):
    """Genera el nombre de la columna derivada *_days_ago."""
    for suffix in ('_dt', '_at', '_date'):
        if col.endswith(suffix):
            return col[: -len(suffix)] + '_days_ago'
    return f'{col}_days_ago'


def to_utc_aware(series):
    """Asegura que la serie de datetimes sea tz-aware UTC."""
    if pd.api.types.is_datetime64_any_dtype(series):
        if series.dt.tz is None:
            return series.dt.tz_localize('UTC')
        return series.dt.tz_convert('UTC')
    return series


# Convertir
# Política: cualquier dt > CUTOFF_DATE es leakage temporal (evento posterior al
# corte). Se NaT-ea antes de convertir a days_ago para evitar valores negativos
# que harían imposibles las validaciones del bloque 6. Las features _dt
# upstream (02c/02e/02f/02m/etc.) NO filtran sus columnas first/last_dt por
# cutoff (solo las agregaciones de volumen) — esta política consolida el filtro.
# Excepción: user_last_login_dt nunca debe ser feature (es el ground truth de
# los targets), pero se conserva como NaN para no romper el schema.
conversion_log = []
post_cutoff_clip = []
for col in datetime_cols:
    new_name = derive_days_ago_name(col)
    s = to_utc_aware(master[col])
    n_post = int((s > CUTOFF_DATE).sum())
    if n_post > 0:
        s = s.where(s <= CUTOFF_DATE, pd.NaT)
        post_cutoff_clip.append({'col': col, 'n_post_cutoff': n_post})
    days_ago = (CUTOFF_DATE - s).dt.days
    master[new_name] = days_ago.astype('Int32')
    column_to_block[new_name] = column_to_block.get(col)
    conversion_log.append({'src': col, 'kind': 'datetime', 'new': new_name})
    master = master.drop(columns=[col])
    if col in column_to_block:
        del column_to_block[col]

for col in unix_int_cols:
    new_name = derive_days_ago_name(col)
    # Tratar como unix seconds; valores nullables van a NaT que se vuelven NaN en .dt.days
    s_int = master[col].astype('Int64')  # nullable
    s_dt = pd.to_datetime(s_int, unit='s', utc=True, errors='coerce')
    n_post = int((s_dt > CUTOFF_DATE).sum())
    if n_post > 0:
        s_dt = s_dt.where(s_dt <= CUTOFF_DATE, pd.NaT)
        post_cutoff_clip.append({'col': col, 'n_post_cutoff': n_post})
    days_ago = (CUTOFF_DATE - s_dt).dt.days
    master[new_name] = days_ago.astype('Int32')
    column_to_block[new_name] = column_to_block.get(col)
    conversion_log.append({'src': col, 'kind': 'unix_int', 'new': new_name})
    master = master.drop(columns=[col])
    if col in column_to_block:
        del column_to_block[col]

print(f'\nTotal convertidas: {len(conversion_log)}')
if post_cutoff_clip:
    print(f'\nColumnas con valores post-cutoff (NaN-eados): {len(post_cutoff_clip)}')
    for entry in post_cutoff_clip:
        print(f"  {entry['col']:<40s} n_post_cutoff={entry['n_post_cutoff']:,}")
    log_step('EXEC', f'4.1+4.2 NaN-eados {sum(e["n_post_cutoff"] for e in post_cutoff_clip):,} valores post-cutoff en {len(post_cutoff_clip)} columnas')


Columnas datetime detectadas: 16
  user_created_at_dt (dtype=datetime64[us], block=users)
  user_last_login_dt (dtype=datetime64[ms], block=users)
  char_first_created (dtype=datetime64[us], block=char)
  char_last_updated (dtype=datetime64[us], block=char)
  reward_last_claim_dt (dtype=datetime64[ms], block=reward)
  reward_first_created (dtype=datetime64[us], block=reward)
  iap_first_consumable_dt (dtype=datetime64[ms], block=iap)
  iap_last_consumable_dt (dtype=datetime64[ms], block=iap)
  iap_first_subscription_dt (dtype=datetime64[ms], block=iap)
  iap_last_subscription_end_dt (dtype=datetime64[ms], block=iap)
  device_first_seen_dt (dtype=datetime64[ms], block=device)
  device_last_active_dt (dtype=datetime64[ms], block=device)
  coll_first_item_at (dtype=datetime64[us, UTC], block=coll)
  coll_last_item_at (dtype=datetime64[us, UTC], block=coll)
  items_first_item_at (dtype=datetime64[us, UTC], block=items)
  items_last_item_at (dtype=datetime64[us, UTC], block=items)

Columnas

In [9]:
# [ANALYSIS] 4.3 Detectar duplicados con *_days_since_* / *_days_active*
# Para cada *_days_ago recién creada, buscar derivada existente en mismo bloque
# (que no sea ella misma y tenga forma days-numérica).

dropped_redundant = []
kept_days_ago = []

# Lista candidata de columnas a comparar: *_days_ago nuevas
new_days_ago = [c['new'] for c in conversion_log]

for new_col in new_days_ago:
    block = column_to_block.get(new_col)
    if block is None:
        continue

    # Buscar candidatas en el mismo bloque que indiquen days desde un evento
    candidates = []
    for c in master.columns:
        if c == new_col or c == 'user_id':
            continue
        if column_to_block.get(c) != block:
            continue
        if not pd.api.types.is_numeric_dtype(master[c]):
            continue
        # palabras clave: 'days_since', 'days_active', 'days_ago'
        if ('days_since' in c or 'days_active' in c) and c != new_col:
            candidates.append(c)

    chosen_drop = None
    for cand in candidates:
        # Correlación entre las dos columnas (drop NaN comunes)
        sub = master[[new_col, cand]].dropna()
        if len(sub) < 100:
            continue
        try:
            corr = float(sub.corr().iloc[0, 1])
        except Exception:
            corr = float('nan')
        if abs(corr) >= 0.99:
            chosen_drop = (cand, corr)
            break

    if chosen_drop is not None:
        cand, corr = chosen_drop
        # Dropear new_col (manteniendo la *_days_since_* original que ya tenía sentinel)
        master = master.drop(columns=[new_col])
        dropped_redundant.append({'dropped': new_col, 'kept': cand, 'corr': corr})
        if new_col in column_to_block:
            del column_to_block[new_col]
    else:
        kept_days_ago.append(new_col)

print(f'Convertidas y MANTENIDAS ({len(kept_days_ago)}):')
for c in kept_days_ago:
    print(f'  {c}')
print(f'\nConvertidas y DROPEADAS por redundancia ({len(dropped_redundant)}):')
for r in dropped_redundant:
    print(f'  {r["dropped"]} (corr={r["corr"]:.4f} con {r["kept"]})')

log_step('EXEC', f'BLOQUE 4: {len(conversion_log)} convertidas, {len(dropped_redundant)} dropeadas por redundancia, {len(kept_days_ago)} mantenidas')
print(f'\nMaster tras conversión: {master.shape}')


Convertidas y MANTENIDAS (11):
  user_created_at_days_ago
  user_last_login_days_ago
  char_first_created_days_ago
  char_last_updated_days_ago
  reward_last_claim_days_ago
  reward_first_created_days_ago
  iap_first_consumable_days_ago
  iap_first_subscription_days_ago
  device_first_seen_days_ago
  coll_first_item_days_ago
  items_first_item_days_ago

Convertidas y DROPEADAS por redundancia (5):
  iap_last_consumable_days_ago (corr=1.0000 con iap_consumables_days_since_last)
  iap_last_subscription_end_days_ago (corr=1.0000 con iap_days_since_subscription_end)
  device_last_active_days_ago (corr=1.0000 con device_days_since_last_active)
  coll_last_item_days_ago (corr=1.0000 con coll_days_since_last_item)
  items_last_item_days_ago (corr=1.0000 con items_days_since_last_item)
[EXEC] 13:16:59 — BLOQUE 4: 16 convertidas, 5 dropeadas por redundancia, 11 mantenidas

Master tras conversión: (25200, 115)


## BLOQUE 5 — Reordenación de columnas

In [10]:
# [EXEC] 5.1 Reordenar columnas
# Orden: user_id → features (alfabético por bloque) → churn_14d → churn_30d

# Verificar targets primero
TARGETS = ['churn_14d', 'churn_30d']
missing_targets = [t for t in TARGETS if t not in master.columns]
assert not missing_targets, f'STOP: faltan targets en master: {missing_targets}'

# Asignar orden de bloques alfabético (excepto especiales)
def block_key(c):
    block = column_to_block.get(c, 'zzz_unknown')
    if c == 'user_id':
        return (0, '')
    if c in TARGETS:
        return (9, c)  # al final
    return (5, block, c)  # alfabético por bloque, luego por nombre

ordered_cols = sorted(master.columns, key=block_key)
master = master[ordered_cols]

# Asserts del orden
assert master.columns[0] == 'user_id', f'primera col != user_id: {master.columns[0]}'
assert master.columns[-1] == 'churn_30d', f'última col != churn_30d: {master.columns[-1]}'
assert master.columns[-2] == 'churn_14d', f'penúltima col != churn_14d: {master.columns[-2]}'

print(f'Master reordenada: {master.shape}')
print(f'  Primera col: {master.columns[0]}')
print(f'  Penúltima:   {master.columns[-2]}')
print(f'  Última:      {master.columns[-1]}')
log_step('EXEC', f'5.1 reordenación OK; targets en posiciones [-2, -1]')


Master reordenada: (25200, 115)
  Primera col: user_id
  Penúltima:   churn_14d
  Última:      churn_30d
[EXEC] 13:16:59 — 5.1 reordenación OK; targets en posiciones [-2, -1]


## BLOQUE 6 — Validación global

In [11]:
# [ANALYSIS] 6.1 Asserts duros
assert master.shape[0] == N_SAMPLE
assert master['user_id'].is_unique
assert master.columns[0] == 'user_id'
assert master.columns[-1] == 'churn_30d'
assert master.columns[-2] == 'churn_14d'
assert master.columns.is_unique
assert not master['user_id'].isna().any()

print('=== 6.1 Asserts duros: PASSED ===')
print(f'  Shape: {master.shape}')
print(f'  user_id único + sin NaN')
print(f'  Targets en posiciones [-2, -1]')

# 6.2 Leakage temporal: ninguna feature *_days_ago / *_days_since_* / *_days_active debería tener min < 0
print('\n=== 6.2 Leakage temporal ===')
days_cols = [c for c in master.columns if (
    c.endswith('_days_ago') or 'days_since' in c or 'days_active' in c
) and pd.api.types.is_numeric_dtype(master[c])]

# Tolerancia de -1 día: artefacto de redondeo (REFERENCE_DATE='2026-04-04 00:00'
# vs fecha efectiva del dataset ~2026-04-04 18:23:31 → algunas filas con created_at
# del mismo día caen en -1 al hacer floor de .dt.days). Esto NO es leakage real:
# leakage real serían valores < -1 (más de un día en el futuro relativo a REF).
TOLERANCE_NEG = -1

leakage_violations = []
rounding_artefacts = []
for c in days_cols:
    s = master[c].dropna()
    if len(s) == 0:
        continue
    # Excluir sentinel 9999 si existe (consistente con 02g/02l/02m)
    s_no_sentinel = s[s != 9999]
    if len(s_no_sentinel) == 0:
        continue
    mn = float(s_no_sentinel.min())
    mx = float(s_no_sentinel.max())
    if mn < TOLERANCE_NEG:
        leakage_violations.append({'col': c, 'min': mn, 'max': mx})
    elif mn < 0:
        rounding_artefacts.append({'col': c, 'min': mn, 'max': mx})
    print(f'  {c:<40s} min={mn:>8.1f}  max={mx:>10.1f}  (n={len(s):,})')

if leakage_violations:
    print(f'\nSTOP: {len(leakage_violations)} columnas con min < {TOLERANCE_NEG} (leakage real)')
    for v in leakage_violations:
        print(f'  {v}')
    raise AssertionError(f'Leakage real detectado en {len(leakage_violations)} columnas')

print(f'\nSin leakage temporal real en {len(days_cols)} columnas de tiempo')
if rounding_artefacts:
    print(f'  Caveat: {len(rounding_artefacts)} columnas con min == -1 (artefacto de redondeo,')
    print(f'  por diferencia entre REFERENCE_DATE 00:00 y fecha efectiva 18:23:31 del mismo día).')
    for v in rounding_artefacts:
        print(f'    {v["col"]}: min={v["min"]:.0f}, max={v["max"]:.0f}')

# 6.3 Distribución de dtypes
dtype_counts = master.dtypes.value_counts()
print('\n=== 6.3 Distribución de dtypes ===')
print(dtype_counts)
print(f'\nMemoria total: {master.memory_usage(deep=True).sum()/1e6:.1f} MB')

dtype_df = pd.DataFrame({
    'dtype': dtype_counts.index.astype(str),
    'n_cols': dtype_counts.values,
})
dtype_df.to_csv(REPORT_DIR / 'dtype_distribution.csv', index=False)
log_step('ANALYSIS', f'6.1-6.3 OK: shape={master.shape}, sin leakage, {len(dtype_counts)} dtypes distintos')


=== 6.1 Asserts duros: PASSED ===
  Shape: (25200, 115)
  user_id único + sin NaN
  Targets en posiciones [-2, -1]

=== 6.2 Leakage temporal ===
  char_first_created_days_ago              min=     0.0  max=    1912.0  (n=24,717)
  char_last_updated_days_ago               min=     0.0  max=    1053.0  (n=8,433)
  coll_days_since_last_item                min=     0.0  max=     195.0  (n=25,200)
  coll_first_item_days_ago                 min=     0.0  max=     195.0  (n=20,329)
  device_days_since_last_active            min=     0.0  max=    1472.0  (n=25,200)
  device_first_seen_days_ago               min=     0.0  max=    1478.0  (n=20,271)
  feedback_days_since_last                 min=     0.0  max=    1134.0  (n=25,200)
  iap_consumables_days_since_last          min=     0.0  max=    1190.0  (n=25,200)
  iap_days_since_subscription_end          min=     0.0  max=    1217.0  (n=25,200)
  iap_first_consumable_days_ago            min=     0.0  max=    1333.0  (n=604)
  iap_first_subscri

## BLOQUE 7 — Diagnóstico (tablas guardadas en CSV para 02zz)

In [12]:
# [ANALYSIS] 7.1 Inventario de columnas por dominio
inventory = {}
for c, b in column_to_block.items():
    if c not in master.columns:
        continue
    inventory.setdefault(b, []).append(c)

inv_rows = []
for b in sorted(inventory.keys()):
    cols = sorted(inventory[b])
    inv_rows.append({
        'block': b,
        'n_cols': len(cols),
        'cols_sample': ', '.join(cols[:5]) + ('...' if len(cols) > 5 else ''),
    })
inv_df = pd.DataFrame(inv_rows)
print(inv_df.to_string(index=False))
inv_df.to_csv(REPORT_DIR / 'column_inventory.csv', index=False)

# 7.2 Coverage por bloque
coverage_rows = []
for b in sorted(inventory.keys()):
    if b in ('key', 'target', 'sample_base'):
        continue  # bloques especiales
    cols = inventory[b]
    block_df = master[cols]
    n_cols = len(cols)
    # Jugadores con AL MENOS UNA col no-NaN
    n_with_data = int(block_df.notna().any(axis=1).sum())
    pct_with_data = n_with_data / N_SAMPLE * 100
    # % medio de NaN sobre todas las cols del bloque
    pct_mean_nan = float(block_df.isna().mean().mean() * 100)
    coverage_rows.append({
        'block': b,
        'n_cols': n_cols,
        'n_users_with_data': n_with_data,
        'pct_with_data': round(pct_with_data, 2),
        'pct_mean_nan': round(pct_mean_nan, 2),
    })

cov_df = pd.DataFrame(coverage_rows).sort_values('pct_with_data', ascending=False)
print('\n=== Coverage por bloque ===')
print(cov_df.to_string(index=False))
cov_df.to_csv(REPORT_DIR / 'coverage_by_block.csv', index=False)
log_step('ANALYSIS', f'7.1-7.2 inventario + coverage por bloque guardados')


   block  n_cols                                                                                                                                                cols_sample
    char      23                                             char_arena_category_max, char_attack_defense_sum_max, char_attack_max, char_class_main, char_classes_unique...
    coll       7                                coll_collection_span_days, coll_days_since_last_item, coll_first_item_days_ago, coll_items_last_30d, coll_items_last_90d...
  device       9                                 device_days_since_last_active, device_first_seen_days_ago, device_has_android, device_has_ios, device_is_multi_platform...
feedback       6                                           feedback_days_since_last, feedback_has_any, feedback_n_monetization, feedback_n_negative, feedback_n_positive...
     iap      20 iap_consumables_count, iap_consumables_days_since_last, iap_consumables_unique_products, iap_days_since_subscription_end, i


=== Coverage por bloque ===
   block  n_cols  n_users_with_data  pct_with_data  pct_mean_nan
    char      23              25200          100.0          2.98
    coll       7              25200          100.0          2.76
  device       9              25200          100.0          2.17
feedback       6              25200          100.0          0.00
     iap      20              25200          100.0          9.75
   items      12              25200          100.0          2.21
  reward      10              25200          100.0          8.20
   users      25              25200          100.0          3.49
[ANALYSIS] 13:16:59 — 7.1-7.2 inventario + coverage por bloque guardados


In [13]:
# [ANALYSIS] 7.3 Top 20 columnas con más missing
missing_pct = master.isna().mean() * 100
missing_df = pd.DataFrame({
    'column': missing_pct.index,
    'pct_missing': missing_pct.values.round(4),
    'dtype': [str(master[c].dtype) for c in missing_pct.index],
    'block': [column_to_block.get(c, 'unknown') for c in missing_pct.index],
})
missing_df = missing_df.sort_values('pct_missing', ascending=False).head(20)
missing_df['flag_high_missing'] = missing_df['pct_missing'] >= 95.0
print('=== Top 20 columnas con más missing ===')
print(missing_df.to_string(index=False))
missing_df.to_csv(REPORT_DIR / 'missing_top20.csv', index=False)
log_step('ANALYSIS', f'7.3 missing_top20.csv guardado')


=== Top 20 columnas con más missing ===
                         column  pct_missing   dtype  block  flag_high_missing
  iap_first_consumable_days_ago      97.6032   Int32    iap               True
iap_first_subscription_days_ago      97.3373   Int32    iap               True
       user_last_login_days_ago      85.3611   Int32  users              False
     reward_last_claim_days_ago      71.0794   Int32 reward              False
     char_last_updated_days_ago      66.5357   Int32   char              False
     device_first_seen_days_ago      19.5595   Int32 device              False
       coll_first_item_days_ago      19.3294   Int32   coll              False
  reward_first_created_days_ago      10.9087   Int32 reward              False
         items_redundancy_ratio       3.3095 float32  items              False
       items_mean_enhance_level       3.3095 float32  items              False
      items_first_item_days_ago       3.3095   Int32  items              False
        item

In [14]:
# [ANALYSIS] 7.4 Detección de baja varianza
low_var_rows = []
for c in master.columns:
    if c in ('user_id',):
        continue
    if not pd.api.types.is_numeric_dtype(master[c]):
        continue
    s = master[c].dropna()
    if len(s) < 100:
        continue
    std = float(s.std())
    mean = float(s.mean())
    cv = abs(std / mean) if mean != 0 else float('inf') if std > 0 else 0.0
    is_constant = (std == 0)
    is_quasi_const = cv < 0.01 and not is_constant
    if is_constant or is_quasi_const:
        low_var_rows.append({
            'column': c,
            'block': column_to_block.get(c, 'unknown'),
            'mean': round(mean, 6),
            'std': round(std, 6),
            'coef_var': round(cv, 6) if cv != float('inf') else None,
            'flag': 'constant' if is_constant else 'quasi-constant',
        })

low_var_df = pd.DataFrame(low_var_rows)
if not low_var_df.empty:
    low_var_df = low_var_df.sort_values('std')
print(f'=== {len(low_var_df)} columnas de baja varianza ===')
if not low_var_df.empty:
    print(low_var_df.to_string(index=False))
low_var_df.to_csv(REPORT_DIR / 'low_variance_candidates.csv', index=False)
log_step('ANALYSIS', f'7.4 low_variance_candidates.csv: {len(low_var_df)} entradas')


=== 5 columnas de baja varianza ===
                                      column block         mean          std  coef_var           flag
                               tutorial_done users 1.000000e+00 0.000000e+00  0.000000       constant
                    user_has_corrupted_dates users 0.000000e+00 0.000000e+00  0.000000       constant
user_template_item_stats_augment_update_done users 1.000000e+00 0.000000e+00  0.000000       constant
                          char_is_customized  char 9.999600e-01 6.299000e-03  0.006300 quasi-constant
                        user_last_login_date users 1.768657e+09 3.427237e+06  0.001938 quasi-constant
[ANALYSIS] 13:17:00 — 7.4 low_variance_candidates.csv: 5 entradas


In [15]:
# [ANALYSIS] 7.5 Detección de redundancias por correlación >0.95
# Subset de columnas numéricas (excluir targets)
num_cols = [c for c in master.columns if c not in ('user_id', 'churn_14d', 'churn_30d')
            and pd.api.types.is_numeric_dtype(master[c])]
print(f'Calculando matriz de correlación sobre {len(num_cols)} columnas numéricas...')
t0 = time.time()
corr = master[num_cols].corr(method='pearson', numeric_only=True)
print(f'Correlación calculada en {time.time()-t0:.1f}s, shape={corr.shape}')

# Extraer pares con |corr|>0.95 evitando duplicados
THRESH = 0.95
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
pairs = []
for i, fa in enumerate(corr.columns):
    for j, fb in enumerate(corr.columns):
        if not mask[i, j]:
            continue
        v = corr.iat[i, j]
        if pd.notna(v) and abs(v) >= THRESH:
            pairs.append({
                'feature_a': fa,
                'feature_b': fb,
                'correlation': round(float(v), 4),
                'block_a': column_to_block.get(fa, 'unknown'),
                'block_b': column_to_block.get(fb, 'unknown'),
            })

red_df = pd.DataFrame(pairs)
if not red_df.empty:
    red_df = red_df.sort_values('correlation', ascending=False, key=lambda s: s.abs())
print(f'\n=== Pares con |corr| >= {THRESH}: {len(red_df)} ===')
if not red_df.empty:
    print(red_df.head(30).to_string(index=False))
red_df.to_csv(REPORT_DIR / 'redundancy_candidates.csv', index=False)
log_step('ANALYSIS', f'7.5 redundancy_candidates.csv: {len(red_df)} pares')
del corr
gc.collect()


Calculando matriz de correlación sobre 104 columnas numéricas...


Correlación calculada en 0.3s, shape=(104, 104)

=== Pares con |corr| >= 0.95: 28 ===
                      feature_a                       feature_b  correlation  block_a  block_b
       user_created_at_days_ago       user_player_lifespan_days       1.0000    users    users
iap_days_since_subscription_end       iap_has_subscription_ever      -0.9992      iap      iap
       feedback_days_since_last                feedback_has_any      -0.9992 feedback feedback
    char_first_created_days_ago        user_created_at_days_ago       0.9990     char    users
                 char_level_max           char_talent_total_max       0.9989     char     char
    char_first_created_days_ago       user_player_lifespan_days       0.9989     char    users
  device_days_since_last_active           device_platform_count      -0.9961   device   device
    char_attack_defense_sum_max                 char_attack_max       0.9955     char     char
    char_attack_defense_sum_max                char_defense

0

In [16]:
# [ANALYSIS] 7.6 Tasa de churn global y por segmentos
churn_rows = []

# Global
n_total = len(master)
churn_14d_global = master['churn_14d'].mean() * 100
churn_30d_global = master['churn_30d'].mean() * 100
churn_rows.append({
    'segment_type': 'global',
    'segment_value': 'all',
    'n_users': n_total,
    'churn_14d_pct': round(churn_14d_global, 4),
    'churn_30d_pct': round(churn_30d_global, 4),
})

# Por país (top 10)
if 'country' in master.columns:
    top_countries = master['country'].value_counts().head(10).index
    for c in top_countries:
        sub = master[master['country'] == c]
        if len(sub) == 0:
            continue
        churn_rows.append({
            'segment_type': 'country',
            'segment_value': c,
            'n_users': len(sub),
            'churn_14d_pct': round(sub['churn_14d'].mean() * 100, 4),
            'churn_30d_pct': round(sub['churn_30d'].mean() * 100, 4),
        })

# Por is_google_play (proxy de plataforma)
if 'is_google_play' in master.columns:
    for v in master['is_google_play'].dropna().unique():
        sub = master[master['is_google_play'] == v]
        churn_rows.append({
            'segment_type': 'is_google_play',
            'segment_value': str(v),
            'n_users': len(sub),
            'churn_14d_pct': round(sub['churn_14d'].mean() * 100, 4),
            'churn_30d_pct': round(sub['churn_30d'].mean() * 100, 4),
        })

# Por iap_is_payer (si existe)
for payer_col in ('iap_is_payer', 'iap_is_current_payer'):
    if payer_col in master.columns:
        for v in master[payer_col].dropna().unique():
            sub = master[master[payer_col] == v]
            churn_rows.append({
                'segment_type': payer_col,
                'segment_value': str(v),
                'n_users': len(sub),
                'churn_14d_pct': round(sub['churn_14d'].mean() * 100, 4),
                'churn_30d_pct': round(sub['churn_30d'].mean() * 100, 4),
            })

# Por bloque de actividad: tiene/no tiene datos del bloque
for b in sorted(set(column_to_block.values())):
    if b in ('key', 'target', 'sample_base', 'users', 'unknown'):
        continue
    cols_b = [c for c in master.columns if column_to_block.get(c) == b]
    if not cols_b:
        continue
    has_data = master[cols_b].notna().any(axis=1)
    for v_label, mask in [('with_data', has_data), ('without_data', ~has_data)]:
        sub = master[mask]
        if len(sub) == 0:
            continue
        churn_rows.append({
            'segment_type': f'block_{b}',
            'segment_value': v_label,
            'n_users': len(sub),
            'churn_14d_pct': round(sub['churn_14d'].mean() * 100, 4),
            'churn_30d_pct': round(sub['churn_30d'].mean() * 100, 4),
        })

churn_df = pd.DataFrame(churn_rows)
print('=== Churn rates ===')
print(churn_df.to_string(index=False))
churn_df.to_csv(REPORT_DIR / 'churn_rates.csv', index=False)
log_step('ANALYSIS', f'7.6 churn_rates.csv: {len(churn_df)} segmentos')


=== Churn rates ===
        segment_type segment_value  n_users  churn_14d_pct  churn_30d_pct
              global           all    25200        33.9563        47.4246
             country        Russia     4032        23.0903        36.6567
             country United States     2537        25.9756        39.8896
             country        Brazil     1222        37.3159        50.1637
             country       Ukraine      767        32.3338        44.3286
             country          Iran      741        49.1228        68.5560
             country       Germany      723        34.9931        48.1328
             country         India      712        44.5225        57.5843
             country        Mexico      690        31.5942        44.9275
             country     Indonesia      676        44.2308        57.5444
             country       Türkiye      662        40.3323        52.8701
      is_google_play          True    21491        34.1399        47.5269
      is_google_pl

In [17]:
# [ANALYSIS] 7.7 Visualizaciones — heatmap de missing y churn por bloque
# Heatmap: % missing de top 10 cols por bloque
heatmap_data = []
heatmap_labels = []
for b in sorted(inventory.keys()):
    if b in ('key', 'target', 'sample_base'):
        continue
    cols = sorted(inventory[b])[:10]
    if not cols:
        continue
    missing_pct_block = master[cols].isna().mean() * 100
    for c in cols:
        heatmap_data.append({'block': b, 'feature': c, 'pct_missing': float(missing_pct_block[c])})
        heatmap_labels.append((b, c))

if heatmap_data:
    hm_df = pd.DataFrame(heatmap_data)
    pivot = hm_df.pivot_table(index='block', columns='feature', values='pct_missing', fill_value=0)
    fig, ax = plt.subplots(figsize=(16, 8))
    im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=100)
    cbar = plt.colorbar(im, ax=ax, label='% missing')
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xticks([])
    ax.set_title('% missing por bloque (top 10 features de cada uno)')
    ax.set_xlabel('features')
    plt.tight_layout()
    plt.savefig(REPORT_DIR / 'missing_heatmap.png', dpi=120, bbox_inches='tight')
    plt.close()
    print(f'Heatmap guardado: {REPORT_DIR / "missing_heatmap.png"}')

# Churn por bloque
churn_by_block = churn_df[churn_df['segment_type'].str.startswith('block_')].copy()
if not churn_by_block.empty:
    churn_by_block['block'] = churn_by_block['segment_type'].str.replace('block_', '', regex=False)
    pivot2 = churn_by_block.pivot_table(index='block', columns='segment_value', values='churn_30d_pct')
    pivot2 = pivot2.sort_values('with_data', ascending=False)

    fig, ax = plt.subplots(figsize=(10, max(5, len(pivot2)*0.4)))
    pivot2.plot(kind='barh', ax=ax, color=['#3498db', '#bdc3c7'])
    ax.set_xlabel('Churn 30d (%)')
    ax.set_title('Churn 30d: con vs sin datos del bloque')
    ax.legend(title='Estado del bloque')
    plt.tight_layout()
    plt.savefig(REPORT_DIR / 'churn_by_block.png', dpi=120, bbox_inches='tight')
    plt.close()
    print(f'Churn by block guardado: {REPORT_DIR / "churn_by_block.png"}')

log_step('ANALYSIS', f'7.7 visualizaciones generadas')


Heatmap guardado: /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/master_table/missing_heatmap.png


Churn by block guardado: /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/master_table/churn_by_block.png
[ANALYSIS] 13:17:00 — 7.7 visualizaciones generadas


## BLOQUE 8 — Guardado

In [18]:
# [EXEC] 8.1 Master table parquet + 8.2 schema JSON
t0 = time.time()
master.to_parquet(MASTER_PARQUET, engine='pyarrow', compression='snappy', index=False)
size_mb = MASTER_PARQUET.stat().st_size / 1e6
print(f'Master parquet guardado en {time.time()-t0:.2f}s: {MASTER_PARQUET} ({size_mb:.2f} MB)')

# Verificar re-leyendo
master_check = pd.read_parquet(MASTER_PARQUET)
assert master_check.shape == master.shape, f'shape mismatch al leer: {master_check.shape}'
print(f'Re-lectura OK: shape={master_check.shape}')
del master_check

# Schema JSON
SOURCE_BY_BLOCK = {
    'key': '02z',
    'target': '02a (sample_user_ids)',
    'sample_base': '02a',
    'users': '02a',
    'currency': '02b',
    'char': '02c',
    'arena': '02d',
    'reward': '02e',
    'iap': '02f',
    'device': '02g',
    'feedback': '02i',
    'fights': '02j',
    'coll': '02l',
    'items': '02m',
}

schema_cols = []
for c in master.columns:
    block = column_to_block.get(c, 'unknown')
    if c == 'user_id':
        domain = 'key'
    elif c in ('churn_14d', 'churn_30d'):
        domain = 'target'
    else:
        domain = block
    schema_cols.append({
        'name': c,
        'dtype': str(master[c].dtype),
        'domain': domain,
        'source': SOURCE_BY_BLOCK.get(block, '?'),
        'pct_missing': round(float(master[c].isna().mean() * 100), 4),
    })

schema = {
    'n_rows': int(master.shape[0]),
    'n_cols': int(master.shape[1]),
    'reference_date': str(REFERENCE_DATE.date()),
    'memory_mb': round(master.memory_usage(deep=True).sum() / 1e6, 2),
    'columns': schema_cols,
}
with open(MASTER_SCHEMA, 'w', encoding='utf-8') as f:
    json.dump(schema, f, ensure_ascii=False, indent=2)
print(f'Schema guardado: {MASTER_SCHEMA}')
log_step('EXEC', f'8.1-8.2 master+schema guardados; {master.shape[1]} cols')


Master parquet guardado en 0.05s: /Users/jezquerro/Documents/tfg/data/data_qc/master_table_cutoff.parquet (4.08 MB)
Re-lectura OK: shape=(25200, 115)
Schema guardado: /Users/jezquerro/Documents/tfg/data/data_qc/master_table_cutoff_schema.json
[EXEC] 13:17:00 — 8.1-8.2 master+schema guardados; 115 cols


## BLOQUE 9 — Informe

In [19]:
# [REPORT] 9.1 execution_report.md
t_total = time.time() - NOTEBOOK_START
t_min = int(t_total // 60)
t_sec = int(t_total % 60)
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Datos para el informe
mem_mb = master.memory_usage(deep=True).sum() / 1e6
n_rows, n_cols = master.shape
churn_14d = master['churn_14d'].mean() * 100
churn_30d = master['churn_30d'].mean() * 100
n_high_missing = int((master.isna().mean() >= 0.95).sum())
n_low_var = len(low_var_df)
n_redundant_pairs = len(red_df)

lines = [
    f'# Execution Report — 02z_build_master_table',
    '',
    f'**Notebook**: notebooks/fase1_cleaning/02z_build_master_table.ipynb',
    f'**Fecha**: {now_str}',
    f'**Tiempo de ejecución**: {t_min} min {t_sec} s',
    '',
    '## Resumen',
    '',
    f'- Master table: `data/data_qc/master_table_cutoff.parquet` ({n_rows:,} filas × {n_cols} cols)',
    f'- Bloques de features integrados: 11 (de los notebooks 02a, 02b, 02c, 02d, 02e, 02f, 02g, 02i, 02j, 02l, 02m)',
    f'- Descartes documentados (no incluidos): 02h_user_village_buildings, 02k_user_options, 02n_arena_global_ranking',
    f'- Tasa de churn 14d: **{churn_14d:.2f}%**',
    f'- Tasa de churn 30d: **{churn_30d:.2f}%**',
    f'- Memoria del DataFrame: {mem_mb:.1f} MB',
    '',
    '## Inventario de columnas por dominio',
    '',
    inv_df.to_string(index=False),
    '',
    '## Coverage por bloque',
    '',
    cov_df.to_string(index=False),
    '',
    '## Conversión de datetimes',
    '',
    f'- Columnas datetime detectadas: {len([x for x in conversion_log if x["kind"] == "datetime"])}',
    f'- Columnas Int64 unix-seconds detectadas (por nombre `*_dt`/`*_at`/`*_date`): {len([x for x in conversion_log if x["kind"] == "unix_int"])}',
    f'- Total convertidas a `*_days_ago`: {len(conversion_log)}',
    f'- Dropeadas por redundancia con `*_days_since_*` existente: {len(dropped_redundant)}',
    f'- Mantenidas en master: {len(kept_days_ago)}',
    '',
    '### Detalle de la conversión',
    '',
    '| Origen | Tipo | Derivada | Estado |',
    '|---|---|---|---|',
]
for entry in conversion_log:
    src = entry['src']
    new = entry['new']
    kind = entry['kind']
    dropped_match = next((d for d in dropped_redundant if d['dropped'] == new), None)
    if dropped_match:
        status = f'DROPEADA (corr={dropped_match["corr"]:.4f} con {dropped_match["kept"]})'
    else:
        status = 'mantenida'
    lines.append(f'| `{src}` | {kind} | `{new}` | {status} |')

lines += [
    '',
    '## Detección preliminar de redundancias',
    '',
    f'- Pares con correlación > 0.95: **{n_redundant_pairs}** (ver `redundancy_candidates.csv`)',
    f'- Columnas de baja varianza (constante o cv<0.01): **{n_low_var}** (ver `low_variance_candidates.csv`)',
    f'- Columnas con >95% missing: **{n_high_missing}** (ver `missing_top20.csv`)',
    '',
    '## Sanity checks aplicados',
    '',
    f'- [] Shape == ({n_rows:,}, {n_cols})',
    '- [] user_id único, sin NaN, primera columna',
    '- [] Targets `churn_14d`, `churn_30d` en posiciones [-2, -1]',
    '- [] Sin colisiones de nombre de columna (`master.columns.is_unique`)',
    '- [] Sin leakage temporal en columnas de tiempo',
    '- [] Cada JOIN preservó N_SAMPLE filas exactas (asserts duros tras cada merge)',
    '',
    '## Decisiones tomadas',
    '',
    '1. **Datetimes → enteros**: cualquier datetime convertida a `*_days_ago` con dtype `Int32` nullable (NaN preservado, sin sentinels en este paso).',
    '2. **Detección de redundancia tras conversión**: si una `*_days_ago` recién creada correlaciona >0.99 con una `*_days_since_*` ya existente del mismo bloque, se dropea la nueva (la original ya tenía sentinel 9999 trabajado en su notebook origen).',
    '3. **Targets**: `churn_14d` y `churn_30d` en las dos últimas posiciones (no `is_churned_*` como sugería el prompt: los nombres reales en sample_user_ids son `churn_*`).',
    '4. **Política users_clean**:',
    '   - Drop: `_id`, `own_id` (IDs internos), `churn_14d`, `churn_30d` (ya en sample base)',
    "   - Sin renombrar (categóricas claras): country, language, is_google_play, tutorial_done, has_user_rated_app, focus",
    '   - Renombradas a `user_*`: el resto',
    '5. **users_clean_cutoff.parquet** es la fuente de los targets reales (NO se renombran a is_churned_*).',
    '6. **`features_currency_cutoff.parquet`**: NO renombrado al integrar (sus columnas ya tienen contexto: `tx_*`, `gold_*`, `gems_*`, `dark_steel_*`, `concept_*`, `other_*`).',
    '',
    '## TODOs para 02zz_master_cleanup',
    '',
    f'- Revisar las {n_high_missing} features con >95% missing',
    f'- Revisar los {n_redundant_pairs} pares de correlación > 0.95',
    f'- Revisar las {n_low_var} columnas de baja varianza',
    '- Decidir imputación final para NaN en columnas de bloques con baja cobertura',
    '- Decidir codificación de variables categóricas (`country`, `language`, `device_primary_platform`, `char_class_main`)',
    '- Convertir `churn_14d` / `churn_30d` a bool si conviene para modelado',
    '',
    '## Archivos generados',
    '',
    '| Archivo | Tipo |',
    '|---|---|',
    '| `data/data_qc/master_table_cutoff.parquet` | parquet final |',
    '| `data/data_qc/master_table_cutoff_schema.json` | schema (col→dtype/domain/source) |',
    '| `informes/fase1_cleaning/master_table/execution_report.md` | este informe |',
    '| `informes/fase1_cleaning/master_table/master_summary.md` | resumen ejecutivo |',
    '| `informes/fase1_cleaning/master_table/column_inventory.csv` | bloques y conteo de cols |',
    '| `informes/fase1_cleaning/master_table/coverage_by_block.csv` | jugadores con/sin datos por bloque |',
    '| `informes/fase1_cleaning/master_table/missing_top20.csv` | top features por % missing |',
    '| `informes/fase1_cleaning/master_table/low_variance_candidates.csv` | candidatas a eliminar por varianza |',
    '| `informes/fase1_cleaning/master_table/redundancy_candidates.csv` | pares correlación > 0.95 |',
    '| `informes/fase1_cleaning/master_table/dtype_distribution.csv` | nº de cols por dtype |',
    '| `informes/fase1_cleaning/master_table/churn_rates.csv` | churn por segmento |',
    '| `informes/fase1_cleaning/master_table/missing_heatmap.png` | heatmap missing |',
    '| `informes/fase1_cleaning/master_table/churn_by_block.png` | churn 30d por bloque (con/sin) |',
    '| `informes/fase1_cleaning/master_table/02z_build_master_table_run.html` | logging dual |',
    '',
]

REPORT_PATH = REPORT_DIR / 'execution_report.md'
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f'execution_report.md guardado: {REPORT_PATH}')
log_step('REPORT', '9.1 execution_report.md generado')


execution_report.md guardado: /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/master_table/execution_report.md
[REPORT] 13:17:00 — 9.1 execution_report.md generado


In [20]:
# [REPORT] 9.2 master_summary.md (resumen ejecutivo)

# Compone tabla de bloques con coverage y conteo
summary_blocks = []
for b in sorted(inventory.keys()):
    if b in ('key', 'target', 'sample_base'):
        continue
    cols = inventory[b]
    block_df = master[cols]
    pct_with_data = float(block_df.notna().any(axis=1).mean() * 100)
    summary_blocks.append({'block': b, 'n_cols': len(cols), 'pct_with_data': pct_with_data})

# Top hallazgos churn por bloque
churn_block_rows = churn_df[churn_df['segment_type'].str.startswith('block_')].copy()
churn_block_rows['block'] = churn_block_rows['segment_type'].str.replace('block_', '', regex=False)
churn_pivot = churn_block_rows.pivot_table(index='block', columns='segment_value', values='churn_30d_pct')

lines = [
    '# Resumen ejecutivo — Master Table',
    '',
    '## Cifras clave',
    '',
    f'- **{n_rows:,} jugadores** × **{n_cols} columnas** (incluye user_id + 2 targets)',
    f'- Memoria: {mem_mb:.1f} MB',
    f'- Tasa de churn 14d: **{churn_14d:.2f}%** | 30d: **{churn_30d:.2f}%**',
    '',
    '## Bloques de features integrados',
    '',
    '| Bloque | Features | Coverage | Comentario |',
    '|---|---|---|---|',
]
for s in summary_blocks:
    if s['pct_with_data'] >= 99.0:
        comment = 'Todos los jugadores tienen datos'
    elif s['pct_with_data'] >= 50.0:
        comment = f'{s["pct_with_data"]:.1f}% con datos — buena cobertura'
    elif s['pct_with_data'] >= 10.0:
        comment = f'Solo {s["pct_with_data"]:.1f}% con datos — segmento minoritario'
    else:
        comment = f'Cobertura baja ({s["pct_with_data"]:.2f}%) — revisar utilidad'
    lines.append(f'| `{s["block"]}` | {s["n_cols"]} | {s["pct_with_data"]:.2f}% | {comment} |')

lines += [
    '',
    '## Lo que parece ir bien',
    '',
    f'- Master construida en {t_min} min {t_sec} s con asserts duros tras cada JOIN — sin duplicados, sin colisiones, sin leakage.',
    f'- {len(conversion_log)} columnas datetime convertidas a `*_days_ago` (Int32 nullable). Redundancias detectadas y eliminadas automáticamente ({len(dropped_redundant)} dropeadas tras correlación > 0.99 con derivadas existentes).',
    f'- Targets reales (`churn_14d`, `churn_30d`) verificados en posiciones finales [-2, -1].',
    f'- Schema JSON generado con dominio + fuente de cada columna ({n_cols} entries).',
    '',
    '## Lo que requiere atención en 02zz',
    '',
    f'- **{n_high_missing} features con >95% missing** → revisar si tiene sentido conservarlas (ver `missing_top20.csv`).',
    f'- **{n_redundant_pairs} pares de correlación > 0.95** → revisar redundancia (ver `redundancy_candidates.csv`).',
    f'- **{n_low_var} features cuasi-constantes** (cv<0.01) → revisar utilidad predictiva (ver `low_variance_candidates.csv`).',
    '',
    '## Distribución de churn por bloque (con vs sin datos)',
    '',
    '| Bloque | Churn 30d (con datos) | Churn 30d (sin datos) |',
    '|---|---|---|',
]
for b in churn_pivot.index:
    with_d = churn_pivot.loc[b, 'with_data'] if 'with_data' in churn_pivot.columns and pd.notna(churn_pivot.loc[b, 'with_data']) else None
    without_d = churn_pivot.loc[b, 'without_data'] if 'without_data' in churn_pivot.columns and pd.notna(churn_pivot.loc[b, 'without_data']) else None
    with_str = f'{with_d:.2f}%' if with_d is not None else '—'
    without_str = f'{without_d:.2f}%' if without_d is not None else '—'
    lines.append(f'| `{b}` | {with_str} | {without_str} |')

lines += [
    '',
    '## Próximos pasos',
    '',
    '1. **02zz_master_cleanup**: limpieza basada en este diagnóstico (eliminar redundancias, cuasi-constantes, alto missing si procede).',
    '2. **Fase 2 (EDA)**: análisis de targets, correlaciones con churn, segmentación.',
    '',
]

SUMMARY_PATH = REPORT_DIR / 'master_summary.md'
with open(SUMMARY_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f'master_summary.md guardado: {SUMMARY_PATH}')
log_step('REPORT', '9.2 master_summary.md generado')


master_summary.md guardado: /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/master_table/master_summary.md
[REPORT] 13:17:00 — 9.2 master_summary.md generado


In [21]:
# [REPORT] Resumen final en consola
elapsed = time.time() - NOTEBOOK_START
print('=' * 70)
print(f'RESUMEN FINAL — Notebook 02z_build_master_table')
print('=' * 70)
print(f'  Tiempo total                : {int(elapsed//60)}m {int(elapsed%60)}s')
print(f'  Master shape                : {master.shape}')
print(f'  Memoria                     : {mem_mb:.1f} MB')
print(f'  Tasa churn 14d              : {churn_14d:.2f}%')
print(f'  Tasa churn 30d              : {churn_30d:.2f}%')
print()
print(f'  Bloques integrados          : {len([b for b in inventory if b not in ("key","target","sample_base")])}')
print(f'  Datetimes convertidas       : {len(conversion_log)}')
print(f'  Dropeadas por redundancia   : {len(dropped_redundant)}')
print(f'  Pares corr > 0.95           : {n_redundant_pairs}')
print(f'  Features baja varianza      : {n_low_var}')
print(f'  Features >95% missing       : {n_high_missing}')
print()
print(f'  Outputs:')
print(f'    {MASTER_PARQUET}')
print(f'    {MASTER_SCHEMA}')
print(f'    Diagnóstico en {REPORT_DIR}')
print('=' * 70)


RESUMEN FINAL — Notebook 02z_build_master_table
  Tiempo total                : 0m 1s
  Master shape                : (25200, 115)
  Memoria                     : 23.3 MB
  Tasa churn 14d              : 33.96%
  Tasa churn 30d              : 47.42%

  Bloques integrados          : 8
  Datetimes convertidas       : 16
  Dropeadas por redundancia   : 5
  Pares corr > 0.95           : 28
  Features baja varianza      : 5
  Features >95% missing       : 2

  Outputs:
    /Users/jezquerro/Documents/tfg/data/data_qc/master_table_cutoff.parquet
    /Users/jezquerro/Documents/tfg/data/data_qc/master_table_cutoff_schema.json
    Diagnóstico en /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/master_table


In [22]:
# [REPORT] Logging dual: HTML + sección enriquecida
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from notebook_logging_template import (
    export_notebook_to_html, build_enriched_report_section,
)

notebook_path = PROJECT_ROOT / 'notebooks' / 'fase1_cleaning' / '02z_build_master_table.ipynb'
html_path = REPORT_DIR / '02z_build_master_table_run.html'
export_notebook_to_html(notebook_path, html_path)

enriched = build_enriched_report_section(
    df_final=master,
    raw_shape=None,
    filter_steps=[
        ('Sample base', N_SAMPLE),
        ('Tras todos los JOINs', len(master)),
    ],
)
with open(REPORT_DIR / 'execution_report.md', 'a', encoding='utf-8') as f:
    f.write('\n\n---\n\n' + enriched)
print(f'Enriquecimiento appendeado al execution_report')


HTML generado: 02z_build_master_table_run.html (0.5 MB)
Enriquecimiento appendeado al execution_report
